# Stage 02 — Data Preparation

Imputation and outlier treatment based on the approved plan from Stage 01.

**Dataset:** `data/loans.csv` (1000 observations, 21 variables)  
**Target:** `Creditability`  
**Actions:** Outlier capping (IQR method) on 3 numeric variables

In [ ]:
import sys
import os
import hashlib
import warnings
warnings.filterwarnings('ignore')

# Use absolute path for project root
PROJECT_ROOT = r'c:/projects/superagent'
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
import pdtoolkit as pdt

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

RUN_DIR = os.path.join(PROJECT_ROOT, 'runs/2026-03-15_201852')
RAW_PATH = os.path.join(PROJECT_ROOT, 'data/loans.csv')
CLEAN_PATH = os.path.join(RUN_DIR, 'data', 'loans_clean.csv')
FIG_DIR = os.path.join(RUN_DIR, 'figures')

# Load raw data
db = pd.read_csv(RAW_PATH)
print(f"Raw dataset: {db.shape[0]} rows, {db.shape[1]} columns")
print(f"Missing values: {db.isnull().sum().sum()}")
print(f"Target default rate: {db['Creditability'].mean():.4f}")

## Pre-Imputation Distributions

Before applying outlier treatment, capture the raw distributions of the three numeric variables.

In [ ]:
# Variables requiring outlier treatment (from Stage 01)
outlier_vars = ['Duration of Credit (month)', 'Credit Amount', 'Age (years)']

# Store pre-imputation distributions
pre_stats = {}
for var in outlier_vars:
    pre_stats[var] = {
        'mean': db[var].mean(),
        'median': db[var].median(),
        'std': db[var].std(),
        'min': db[var].min(),
        'max': db[var].max(),
        'q25': db[var].quantile(0.25),
        'q75': db[var].quantile(0.75),
        'values': db[var].copy()
    }
    q25, q75 = db[var].quantile(0.25), db[var].quantile(0.75)
    iqr = q75 - q25
    upper_fence = q75 + 1.5 * iqr
    n_outliers = (db[var] > upper_fence).sum()
    print(f"{var}: mean={pre_stats[var]['mean']:.2f}, median={pre_stats[var]['median']:.2f}, "
          f"max={pre_stats[var]['max']:.2f}, IQR upper fence={upper_fence:.2f}, outliers={n_outliers}")

## Outlier Imputation

Apply IQR-based capping using `pdt.imp_outliers()` on the three numeric variables. Only numeric columns are passed to the function.

In [ ]:
# Apply outlier imputation using pdt.imp_outliers on numeric columns only
numeric_df = db[outlier_vars].copy()
imputed_df, outlier_report = pdt.imp_outliers(numeric_df, method='iqr', range_val=1.5)

print("Outlier Imputation Report:")
print(outlier_report.to_string())

# Update the main dataframe with imputed values
db_clean = db.copy()
for var in outlier_vars:
    db_clean[var] = imputed_df[var]

# Track changes per variable
transformations = []
for var in outlier_vars:
    n_changed = (db[var] != db_clean[var]).sum()
    transformations.append({
        'variable': var,
        'type': 'outlier_imputation',
        'method': 'iqr',
        'n_values_changed': int(n_changed),
        'cap_rate_pct': round(100 * n_changed / len(db), 1),
        'cap_value': db_clean[var].max()
    })
    print(f"\n{var}: {n_changed} values capped ({100*n_changed/len(db):.1f}%), "
          f"new max = {db_clean[var].max():.2f}")

## Before/After Distribution Plots

Side-by-side histograms showing the effect of outlier capping on each variable.

In [ ]:
# Generate before/after distribution overlay plots for each imputed variable
BLUE = '#2166AC'
RED = '#D6604D'
GREY = '#999999'

for var in outlier_vars:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Left panel: histogram overlay
    raw_vals = pre_stats[var]['values']
    clean_vals = db_clean[var]
    
    bins = np.linspace(min(raw_vals.min(), clean_vals.min()),
                       raw_vals.max(), 40)
    
    axes[0].hist(raw_vals, bins=bins, alpha=0.5, color=RED, label='Before (raw)', edgecolor='white')
    axes[0].hist(clean_vals, bins=bins, alpha=0.5, color=BLUE, label='After (capped)', edgecolor='white')
    
    # Mark the cap value
    cap_val = clean_vals.max()
    axes[0].axvline(cap_val, color='black', linestyle='--', linewidth=1.5, label=f'Cap = {cap_val:.1f}')
    axes[0].set_title(f'{var} — Histogram Overlay', fontsize=12, fontweight='bold')
    axes[0].set_xlabel(var)
    axes[0].set_ylabel('Frequency')
    axes[0].legend(fontsize=9)
    
    # Right panel: box plots before/after
    box_data = [raw_vals, clean_vals]
    bp = axes[1].boxplot(box_data, labels=['Before', 'After'], patch_artist=True,
                         widths=0.5)
    bp['boxes'][0].set_facecolor(RED)
    bp['boxes'][0].set_alpha(0.5)
    bp['boxes'][1].set_facecolor(BLUE)
    bp['boxes'][1].set_alpha(0.5)
    axes[1].set_title(f'{var} — Box Plot Comparison', fontsize=12, fontweight='bold')
    axes[1].set_ylabel(var)
    
    plt.tight_layout()
    safe_name = var.lower().replace(' ', '_').replace('(', '').replace(')', '')
    fig_path = os.path.join(FIG_DIR, f'02_imputation_{safe_name}.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {fig_path}")

## Post-Imputation Validation

Verify data integrity: no new missing values, row count preserved, distributions plausible.

In [ ]:
# Post-imputation validation
print("=== Post-Imputation Validation ===\n")

# 1. Row count preserved
assert db_clean.shape[0] == db.shape[0], "Row count mismatch!"
assert db_clean.shape[1] == db.shape[1], "Column count mismatch!"
print(f"[PASS] Row count preserved: {db_clean.shape[0]} rows, {db_clean.shape[1]} columns")

# 2. No new missing values
nan_count = db_clean.isnull().sum().sum()
assert nan_count == 0, f"New NaN values introduced: {nan_count}"
print(f"[PASS] No missing values: {nan_count} NaN total")

# 3. Distribution plausibility — compare means
print(f"\n{'Variable':<35} {'Raw Mean':>10} {'Clean Mean':>11} {'Shift %':>8}")
print("-" * 70)
for var in outlier_vars:
    raw_mean = pre_stats[var]['mean']
    clean_mean = db_clean[var].mean()
    shift_pct = 100 * (clean_mean - raw_mean) / raw_mean
    print(f"{var:<35} {raw_mean:>10.2f} {clean_mean:>11.2f} {shift_pct:>7.1f}%")

# 4. Cap rates are reasonable (< 20%)
print(f"\n{'Variable':<35} {'Cap Rate':>10}")
print("-" * 50)
for t in transformations:
    status = "OK" if t['cap_rate_pct'] < 20 else "FLAG"
    print(f"{t['variable']:<35} {t['cap_rate_pct']:>8.1f}%  [{status}]")

# 5. Target variable unchanged
assert (db_clean['Creditability'] == db['Creditability']).all(), "Target variable was modified!"
print(f"\n[PASS] Target variable unchanged (default rate: {db_clean['Creditability'].mean():.4f})")

## Save Clean Dataset

In [ ]:
# Save clean dataset
db_clean.to_csv(CLEAN_PATH, index=False)
print(f"Clean dataset saved to: {CLEAN_PATH}")
print(f"Shape: {db_clean.shape}")

# Compute MD5 checksum
md5_hash = hashlib.md5(open(CLEAN_PATH, 'rb').read()).hexdigest()
print(f"MD5 checksum: {md5_hash}")

In [ ]:
# Write stage_02.md
import json

stage_md_lines = [
    f"clean_dataset_path: {CLEAN_PATH}",
    f"clean_dataset_checksum: {md5_hash}",
    f"n_observations: {db_clean.shape[0]}",
    "transformations_applied:"
]

for t in transformations:
    stage_md_lines.append(f"  - variable: {t['variable']}")
    stage_md_lines.append(f"    type: {t['type']}")
    stage_md_lines.append(f"    method: {t['method']}")
    stage_md_lines.append(f"    n_values_changed: {t['n_values_changed']}")
    stage_md_lines.append(f"    cap_value: {t['cap_value']:.1f}")
    stage_md_lines.append(f"    note: Upper-tail IQR capping ({t['cap_rate_pct']}% of observations)")

# Determine flags
flags = []
for t in transformations:
    if t['cap_rate_pct'] >= 20:
        flags.append(f"{t['variable']} has high cap rate ({t['cap_rate_pct']}%)")

if flags:
    stage_md_lines.append(f"data_preparation_flags:")
    for f in flags:
        stage_md_lines.append(f"  - {f}")
else:
    stage_md_lines.append("data_preparation_flags: None")

stage_md_content = "\n".join(stage_md_lines) + "\n"

stage_md_path = os.path.join(RUN_DIR, 'pipeline', 'stage_02.md')
with open(stage_md_path, 'w') as f:
    f.write(stage_md_content)
print(f"Stage summary written to: {stage_md_path}")
print()
print(stage_md_content)

In [ ]:
# Write stage_02_fixes.md — fix-proposer smoke tests
fixes = []
fix_critical = 0

# 1. Row count preserved
if db_clean.shape[0] != db.shape[0]:
    fixes.append({'severity': 'critical', 'issue': 'Row count changed after imputation',
                  'detail': f'Raw: {db.shape[0]}, Clean: {db_clean.shape[0]}'})
    fix_critical += 1

# 2. No new NaN
if db_clean.isnull().sum().sum() > 0:
    fixes.append({'severity': 'critical', 'issue': 'New NaN values introduced',
                  'detail': f'{db_clean.isnull().sum().sum()} NaN values found'})
    fix_critical += 1

# 3. Outlier cap rate reasonable
for t in transformations:
    if t['cap_rate_pct'] > 20:
        fixes.append({'severity': 'critical', 'issue': f"High cap rate for {t['variable']}",
                      'detail': f"{t['cap_rate_pct']}% of values capped (threshold: 20%)"})
        fix_critical += 1

# 4. Imputation methods documented
undocumented = [t for t in transformations if not t.get('method')]
if undocumented:
    fixes.append({'severity': 'medium', 'issue': 'Undocumented imputation methods',
                  'detail': f"{len(undocumented)} variables without method specification"})

# 5. Mean shift check (advisory)
for var in outlier_vars:
    raw_mean = pre_stats[var]['mean']
    clean_mean = db_clean[var].mean()
    shift_pct = abs(100 * (clean_mean - raw_mean) / raw_mean)
    if shift_pct > 10:
        fixes.append({'severity': 'medium', 'issue': f"Large mean shift for {var}",
                      'detail': f"Mean shifted by {shift_pct:.1f}% after capping"})

fixes_md_lines = ["# Stage 02 Fix-Proposer Diagnostic", ""]
if not fixes:
    fixes_md_lines.append("No issues found. All smoke tests passed.")
else:
    for fix in fixes:
        fixes_md_lines.append(f"- [{fix['severity'].upper()}] {fix['issue']}: {fix['detail']}")

fixes_md_lines.append("")
fixes_md_lines.append(f"Total issues: {len(fixes)} ({fix_critical} critical)")

fixes_path = os.path.join(RUN_DIR, 'pipeline', 'stage_02_fixes.md')
with open(fixes_path, 'w') as f:
    f.write("\n".join(fixes_md_lines) + "\n")
print(f"Fix proposals written to: {fixes_path}")
print(f"Total: {len(fixes)} issues, {fix_critical} critical")

## Summary

**Stage 02 — Data Preparation** completed successfully.

### Actions Performed
- **No special case imputation required** — the dataset has no missing or infinite values
- **Outlier capping (IQR method, 1.5x multiplier)** applied to 3 numeric variables:
  - `Duration of Credit (month)` — upper-tail outliers capped
  - `Credit Amount` — upper-tail outliers capped
  - `Age (years)` — upper-tail outliers capped

### Validation Results
- Row count preserved (1000 observations)
- No new missing values introduced
- All cap rates below 20% threshold
- Target variable unchanged
- Post-imputation distributions are plausible (no suspicious spikes from mean imputation — IQR capping was used)

### Output Files
- Clean dataset: `runs/2026-03-15_201852/data/loans_clean.csv`
- Stage summary: `runs/2026-03-15_201852/pipeline/stage_02.md`
- Fix proposals: `runs/2026-03-15_201852/pipeline/stage_02_fixes.md`
- Figures: `runs/2026-03-15_201852/figures/02_imputation_*.png`